# Введение

Подключение диска

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Загрузка видеозаписи и извлечение из неё кадров и аудио

# Видеомодель

Подключение необходимых библиотек

In [ ]:
import random
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import torchvision
from torchvision import transforms

from PIL import Image
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split, ConcatDataset
from tqdm.notebook import tqdm

from torch import nn
from torchsummary import summary
import torch.nn.functional as F


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

Класс для работы с датасетом

In [ ]:
from PIL import Image
from typing import Tuple

class ImageDataset:
    def __init__(self, root_dir, transform=None):
        """
        Инициализация класса.

        :param root_dir: Путь к папке с подкаталогами, где каждый подкаталог представляет класс и содержит изображения.
        """
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []

        # Сканируем папку и создаём список изображений и их классов

        for image_name in os.listdir(root_dir):
            image_path = os.path.join(root_dir, image_name)
            self.image_paths.append(image_path)

    def __len__(self) -> int:
        """Возвращает количество изображений в датасете."""
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[Image.Image, int]:
        """
        Возвращает изображение и номер его класса по индексу.

        :param idx: Индекс изображения.
        :return: Кортеж (изображение, номер класса).
        """
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image

dir = '/content/output/frames'

Архитектура видеомодели

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(SEBlock, self).__init__()
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(in_channels, in_channels // reduction)
        self.fc2 = nn.Linear(in_channels // reduction, in_channels)

    def forward(self, x):
        batch, channels, _, _ = x.size()
        y = self.global_pool(x).view(batch, channels)
        y = F.relu(self.fc1(y))
        y = torch.sigmoid(self.fc2(y)).view(batch, channels, 1, 1)
        return x * y


class SkipConnectionBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(SkipConnectionBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.dropout1 = nn.Dropout2d(0.3)  # Добавлено dropout

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.dropout2 = nn.Dropout2d(0.3)  # Добавлено dropout

        self.se = SEBlock(out_channels)  # Добавляем SEBlock

        # Используем skip_conv только если количество каналов не совпадает
        self.use_skip_conv = in_channels != out_channels
        if self.use_skip_conv:
            self.skip_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        identity = self.skip_conv(x) if self.use_skip_conv else x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.dropout1(out)  # Dropout после первого слоя
        out = self.bn2(self.conv2(out))
        out = self.dropout2(out)  # Dropout после второго слоя
        out = self.se(out)  # Применяем SEBlock
        out += identity
        return F.relu(out)


class MyModel(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            SkipConnectionBlock(3, 16),
            nn.MaxPool2d(2, 2),

            SkipConnectionBlock(16, 64),
            nn.MaxPool2d(2, 2),

            SkipConnectionBlock(64, 128),
            nn.MaxPool2d(2, 2),

            SkipConnectionBlock(128, 256),
            nn.MaxPool2d(2, 2),

            SkipConnectionBlock(256, 512),
            nn.MaxPool2d(2, 2),

            SkipConnectionBlock(512, 1024),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(1024, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x



model = MyModel(num_classes=6).to(device)
summary(model, (3, 180, 180))

Загрузка кадров как датасета

Инференс видеомодели и получение числовых характеристик предсказаний, а также их нормализация

# Аудио

Подключение необходимых библиотек

In [ ]:
import random
from pathlib import Path
import os
import librosa

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import torchvision
from torchvision import transforms

from PIL import Image
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split, ConcatDataset
from tqdm.notebook import tqdm

from torch import nn
from torchsummary import summary
import torch.nn.functional as F

from keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from glob import glob

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

Константы для работы с аудио

In [ ]:
path = '/content/output/'
name = 'audio.wav'
sr = 16000
dt = 1.0
N_CLASSES = 6
n_mels = 64
hop_length = 256
n_fft = 1024
threshold=0.01
frames = (sr//hop_length)+1

Обработка аудио

In [ ]:
def envelope(y):
    mask = []
    y1 = pd.Series(y).apply(np.abs)
    y_mean = y1.rolling(window=int(sr/20),
                       min_periods=1,
                       center=True).max()
    for mean in y_mean:
        if mean > threshold:
            mask.append(True)
        else:
            mask.append(False)
    return mask

In [ ]:
def load_audio(path, sr):
    rate = librosa.get_samplerate(path)
    y1, _ = librosa.load(path, sr=rate)
    wav = librosa.resample(y=y1, orig_sr=rate, target_sr=sr)
    return wav

In [ ]:
def get_melspectogram(waveform):
    mel_spectrogram = librosa.feature.melspectrogram(y=waveform, sr=sr, n_fft=n_fft,
                                                     hop_length=hop_length, n_mels=n_mels)
    log_mel_spectrogram = librosa.power_to_db(mel_spectrogram)
    return log_mel_spectrogram

In [ ]:
class AudDataset:
    def __init__(self, root_dir, wav_name=None, n_classes=6):
        self.root_dir = root_dir
        self.sr = sr
        self.dt = dt
        self.n_mels = n_mels
        self.frames = frames

        self.wav_paths = []

        if (wav_name):
            wav_path = os.path.join(root_dir, wav_name)
            self.wav_paths.append(wav_path)
        else:
            for wavname in os.listdir(root_dir):
                wav_path = os.path.join(root_dir, wavname)
                self.wav_paths.append(wav_path)


        self.X = self._prepare_data()

    def _prepare_data(self):
        X = np.empty((1, int(self.sr * self.dt)), dtype=np.float32)

        for file in self.wav_paths:
            y1 = load_audio(file, self.sr)
            mask = envelope(y1)
            y = y1[mask]
            n = y.shape[0]
            m = n // self.sr
            for k in range(m):
                a = np.expand_dims(y[k * self.sr:(k + 1) * self.sr], axis=0)
                X = np.append(X, a, axis=0)

        X = np.delete(X, 0, axis=0)

        n_samples = X.shape[0]
        X_mel = np.empty((n_samples, self.n_mels, self.frames), dtype=np.float32)

        for i in np.arange(n_samples):
            X_mel[i] = get_melspectogram(X[i])

        return X_mel

    def __getitem__(self, idx):
        return self.X[idx]

    def __len__(self):
        return len(self.X)

Архитектура аудиомодели

In [ ]:
class Conv2DModel(nn.Module):
    def __init__(self, n_classes):
        super(Conv2DModel, self).__init__()

        # Нормализация слоев
        self.layer_norm = nn.LayerNorm([n_mels, frames])

        # Сверточные слои
        self.conv1 = nn.Conv2d(1, 8, kernel_size=(7, 7), padding='same')
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 2), padding=0)

        self.conv2 = nn.Conv2d(8, 16, kernel_size=(5, 5), padding='same')
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 2), padding=0)

        self.conv3 = nn.Conv2d(16, 16, kernel_size=(3, 3), padding='same')
        self.pool3 = nn.MaxPool2d(kernel_size=(2, 2), padding=0)

        self.conv4 = nn.Conv2d(16, 32, kernel_size=(3, 3), padding='same')
        self.pool4 = nn.MaxPool2d(kernel_size=(2, 2), padding=0)

        self.conv5 = nn.Conv2d(32, 32, kernel_size=(3, 3), padding='same')

        # Полносвязные слои
        self.dropout = nn.Dropout(0.2)
        self.fc1 = nn.Linear(32 * (n_mels // 16) * (frames // 16), 64)
        self.fc2 = nn.Linear(64, n_classes)

    def forward(self, x):
        # Применение LayerNorm к формату последнего канала и восстановление формата первого канала
        x = self.layer_norm(x.squeeze(1)).unsqueeze(1)

        x = torch.tanh(self.conv1(x))
        x = self.pool1(x)

        x = F.relu(self.conv2(x))
        x = self.pool2(x)

        x = F.relu(self.conv3(x))
        x = self.pool3(x)

        x = F.relu(self.conv4(x))
        x = self.pool4(x)

        x = F.relu(self.conv5(x))

        x = torch.flatten(x, start_dim=1)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return F.softmax(x, dim=1)


model = Conv2DModel(n_classes=6).to(device)

summary(model, (64, 63))

# Новый раздел

In [ ]:
import os
from PIL import Image
from typing import Tuple

class ImageDataset:
    def __init__(self, root_dir, transform=None):
        """
        Инициализация класса.

        :param root_dir: Путь к папке с подкаталогами, где каждый подкаталог представляет класс и содержит изображения.
        """
        self.root_dir = root_dir
        self.transform = transform
        self.class_to_idx = {}
        self.image_paths = []
        self.labels = []

        # Сканируем папку и создаём список изображений и их классов
        for class_idx, class_name in enumerate(sorted(os.listdir(root_dir))):
            class_path = os.path.join(root_dir, class_name)
            if os.path.isdir(class_path):
                self.class_to_idx[class_name] = class_idx
                for image_name in os.listdir(class_path):
                    image_path = os.path.join(class_path, image_name)
                    self.image_paths.append(image_path)
                    self.labels.append(class_idx)

    def __len__(self) -> int:
        """Возвращает количество изображений в датасете."""
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[Image.Image, int]:
        """
        Возвращает изображение и номер его класса по индексу.

        :param idx: Индекс изображения.
        :return: Кортеж (изображение, номер класса).
        """
        image_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

dir = '/content/drive/MyDrive/datasets/Accords/'

BATCH_SIZE = 16


transform_orgin = transforms.Compose([
    transforms.Resize((180, 180)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

valid_dataset = ImageDataset(root_dir=dir, transform=transform_orgin)

In [ ]:
train_path = "/content/drive/MyDrive/datasets/Chords/Training"
test_path = "/content/drive/MyDrive/datasets/Chords/Test"

sr = 16000
dt = 1.0
N_CLASSES = 6
n_mels = 64
hop_length = 256
n_fft = 1024
threshold=0.01
frames = (sr//hop_length)+1

def envelope(y):
    mask = []
    y1 = pd.Series(y).apply(np.abs)
    y_mean = y1.rolling(window=int(sr/20),
                       min_periods=1,
                       center=True).max()
    for mean in y_mean:
        if mean > threshold:
            mask.append(True)
        else:
            mask.append(False)
    return mask

def load_audio(path, sr):
    rate = librosa.get_samplerate(path)
    y1, _ = librosa.load(path, sr=rate)
    wav = librosa.resample(y=y1, orig_sr=rate, target_sr=sr)
    return wav

def get_melspectogram(waveform):
    mel_spectrogram = librosa.feature.melspectrogram(y=waveform, sr=sr, n_fft=n_fft,
                                                     hop_length=hop_length, n_mels=n_mels)
    log_mel_spectrogram = librosa.power_to_db(mel_spectrogram)
    return log_mel_spectrogram

class AudDataset:
    def __init__(self, path, n_classes=6):
        self.path = path
        self.sr = sr
        self.dt = dt
        self.n_mels = n_mels
        self.frames = frames
        self.n_classes = n_classes

        self.wav_paths = glob('{}/**'.format(self.path), recursive=True)
        self.wav_paths = [x.replace(os.sep, '/') for x in self.wav_paths if '.wav' in x]

        self.classes = sorted(os.listdir(self.path))
        self.le = LabelEncoder()
        self.le.fit(self.classes)

        self.labels = [os.path.split(x)[0].split('/')[-1] for x in self.wav_paths]
        self.labels = self.le.transform(self.labels)

        self.X, self.Y = self._prepare_data()

    def _prepare_data(self):
        X = np.empty((1, int(self.sr * self.dt)), dtype=np.float32)
        Y = []

        for file, label in zip(self.wav_paths, self.labels):
            y1 = load_audio(file, self.sr)
            mask = envelope(y1)
            y = y1[mask]
            n = y.shape[0]
            m = n // self.sr
            for k in range(m):
                a = np.expand_dims(y[k * self.sr:(k + 1) * self.sr], axis=0)
                X = np.append(X, a, axis=0)
                Y.append(label)

        X = np.delete(X, 0, axis=0)
        Y = np.array(Y, dtype=np.int32)

        n_samples = X.shape[0]
        X_mel = np.empty((n_samples, self.n_mels, self.frames), dtype=np.float32)

        for i in np.arange(n_samples):
            X_mel[i] = get_melspectogram(X[i])

        return X_mel, Y

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

    def __len__(self):
        return len(self.X)

audio_valid_dataset = AudDataset(path=train_path)

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from collections import defaultdict
import itertools

# Устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Модель для изображений
model = MyModel(num_classes=6).to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/best_model1.pth'))
model.eval()

# Модель для аудио
audio_model = Conv2DModel(n_classes=6).to(device)
audio_model.load_state_dict(torch.load('/content/drive/MyDrive/best_model_aud.pth'))
audio_model.eval()

# Загрузчики данных
valid_dataloader = DataLoader(valid_dataset, batch_size=16, shuffle=False)  # Изображения
audio_valid_dataloader = DataLoader(audio_valid_dataset, batch_size=16, shuffle=False)  # Аудио

# Предсказания для изображений
image_predictions = []
image_true_labels = []
with torch.no_grad():
    for images, labels in valid_dataloader:
        images = images.to(device)
        outputs = torch.softmax(model(images), dim=1)
        image_predictions.append(outputs.cpu().numpy())
        image_true_labels.append(labels.numpy())
image_predictions = np.concatenate(image_predictions, axis=0)
image_true_labels = np.concatenate(image_true_labels, axis=0)

# Предсказания для аудио
audio_predictions = []
audio_true_labels = []
with torch.no_grad():
    for audio, labels in audio_valid_dataloader:
        audio = audio.to(device)
        outputs = audio_model(audio)
        audio_predictions.append(outputs.cpu().numpy())
        audio_true_labels.append(labels.numpy())
audio_predictions = np.concatenate(audio_predictions, axis=0)
audio_true_labels = np.concatenate(audio_true_labels, axis=0)

# Группировка предсказаний по классам
image_predictions_by_class = defaultdict(list)
for pred, label in zip(image_predictions, image_true_labels):
    image_predictions_by_class[label].append(pred)

audio_predictions_by_class = defaultdict(list)
for pred, label in zip(audio_predictions, audio_true_labels):
    audio_predictions_by_class[label].append(pred)

# Создание итогового датасета
ensemble_dataset = []
for class_label in range(6):
    image_preds = image_predictions_by_class[class_label]
    audio_preds = audio_predictions_by_class[class_label]
    if not image_preds or not audio_preds:
        continue
    for image_pred, audio_pred in itertools.product(image_preds, audio_preds):
        combined_pred = np.concatenate([image_pred, audio_pred])  # Конкатенация вероятностей
        ensemble_dataset.append(np.append(combined_pred, class_label))

ensemble_dataset = np.array(ensemble_dataset)

# Сохранение
np.save('/content/drive/MyDrive/ensemble_dataset.npy', ensemble_dataset)
print("Итоговый датасет сохранён как ensemble_dataset.npy")

In [ ]:
image_pred

# Обучение

In [ ]:
import random
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import torchvision
from torchvision import transforms

from PIL import Image
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, Dataset, random_split, ConcatDataset
from tqdm.notebook import tqdm

from torch import nn
from torchsummary import summary
import torch.nn.functional as F


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
ensemble_dataset = np.load('/content/drive/MyDrive/ensemble_dataset.npy')
X = ensemble_dataset[:, :-1]  # Первые 12 столбцов — предсказания
y = ensemble_dataset[:, -1].astype(int)  # Последний столбец — метка класса

# 2. Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Преобразование данных в тензоры PyTorch
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# 4. Создание DataLoader для обучения
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_dataloader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

valid_dataset = TensorDataset(X_test_tensor, y_test_tensor)
valid_dataloader = DataLoader(valid_dataset, batch_size=1024, shuffle=False)

# 5. Определение архитектуры модели

class MetaModel(nn.Module):
    def __init__(self, input_size=12, num_classes=6):
        super(MetaModel, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)  # Скрытый слой
        self.relu = nn.ReLU()  # Активация
        self.fc2 = nn.Linear(64, num_classes)  # Выходной слой

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = MetaModel().to(device)
summary(model, (12, ))

In [ ]:


optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()



train_losses = []
val_losses = []

train_acc_scores = []
val_acc_scores = []

best_val_acc = 0.0
best_model_path = '/content/drive/MyDrive/best_metamodel.pth' # Сохраняет модель на диск, что бы не потерять при закончившихся ресурсах колаба

num_epochs = 10

In [ ]:
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    train_true = []
    train_pred = []

    for batch in tqdm(train_dataloader):
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        train_true.extend(labels.cpu().numpy())
        train_pred.extend(preds.cpu().numpy())

    train_acc = accuracy_score(train_true, train_pred)
    train_losses.append(running_loss / len(train_dataloader))
    train_acc_scores.append(train_acc)


    model.eval()
    val_running_loss = 0.0
    val_true = []
    val_pred = []

    # валидационный цикл, когда мы оцениваем качество работы модели на отложенной выборке
    with torch.no_grad():
        for batch in tqdm(valid_dataloader):
            inputs, labels = batch
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            val_true.extend(labels.cpu().numpy())
            val_pred.extend(preds.cpu().numpy())

    val_acc = accuracy_score(val_true, val_pred)
    val_losses.append(val_running_loss / len(valid_dataloader))
    val_acc_scores.append(val_acc)

    # если получившаяся модель лучше предыдущей, сохраним чекпоинт
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f'New best model saved with Accuracy: {best_val_acc:.4f}')


    # выведем в консоль получившиеся результаты на отдельной эпохе
    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {train_losses[-1]:.4f}, Train Accuracy: {train_acc:.4f}, '
          f'Val Loss: {val_losses[-1]:.4f}, Val Accuracy: {val_acc:.4f}')

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_acc_scores, label='Train Accuracy')
plt.plot(val_acc_scores, label='Validation Accuracy')
plt.title('Accuracy Score')
plt.xlabel('Epoch')
plt.ylabel('Accuracy Score')
plt.legend()

plt.show()

In [ ]:
torch.save(model.state_dict(), best_model_path)